# Prompt engineering techniques

#### 1. Initial Configurations & scaffolding

In [5]:
from anthropic.types import model
from anthropic import Anthropic
from dotenv import load_dotenv
from anthropic.resources import messages
import json
from traceback import print_exc
from statistics import mean
import re
import ast

load_dotenv()

True

In [6]:
def add_user_message(messages, text):
    message = {
            "role":"user",
            "content":text
        }
    
    messages.append(message)

def add_assistant_message(messages, text):
    message = {
            "role":"assistant",
            "content":text
        }
    
    messages.append(message)

def chat(messages, system=None, temperature=0.0, stop_sequences=None, model = "claude-haiku-4-5"):

    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)

    return message.content[0].text

#### 2. Generate Test Data Set using Haiku

In [7]:
client = Anthropic()
model = "claude-haiku-4-5"

prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that Write a compact, concise 1 day meal plan for a single athlete. Generate an array of JSON objects,
each representing input & solution criteria.

Example output:
```json
[
    {
        "prompt_input": {
            "height": "Athlete's height in cm",
            "weight": "Athlete's weight in kg", 
            "goal": "Goal of the athlete",
            "restrictions": "Dietary restrictions of the athlete"
        },
        "solution_criteria" : []
    }
]
```

Please generate 3 objects.
"""

messages = []
add_user_message(messages, prompt)
add_assistant_message(messages, "```json")
text = chat(messages, stop_sequences=["```"])

with open('ex04_dataset.json', 'w') as f:
    json.dump(json.loads(text), f, indent=2)

#### 3. Running the eval & Model based grading

In [ ]:
model = "claude-haiku-4-5"

def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Generate meal plan for single single athlete by considering athlete information.

<athlete_information>
    - Height: {test_case["prompt_input"]["height"]}
    - Weight: {test_case["prompt_input"]["weight"]}
    - Goal: {test_case["prompt_input"]["goal"]}
    - Dietary restrictions: {test_case["prompt_input"]["restrictions"]}
</athlete_information>

You can follow the following guidelines:
<guidelines>
      - Meal plan should contain 4-5 meals distributed throughout the day
      - Meals should be practical and quick to prepare
      - Should include pre and post-workout nutrition
      - Format should be compact and easy to read
      - Should be thoughtful about dietary restrictions
</guidelines>

Sample good plans:
<example_plan>
    <plan>
        # 7-Day Muscle Gain Meal Plan for 75kg Athlete**Daily Caloric Target:** ~3,000-3,200 kcal | **Protein:** ~150g | **Carbs:** ~375g | **Fat:** ~85g\n\n---\n\n## **DAILY MEAL STRUCTURE**\n\n### **Meal 1: Breakfast (7:00 AM)**\n- 3 whole eggs + 2 egg whites\n- 80g oatmeal with banana\n- 1 tbsp peanut butter\n- *~650 kcal | 25g protein*\n\n### **Meal 2: Mid-Morning Snack (10:00 AM)**\n- Greek yogurt (200g)\n- Granola (40g)\n- Berries (100g)\n- *~350 kcal | 20g protein*\n\n### **Meal 3: Pre-Workout (1:00 PM)**\n- Chicken breast (150g)\n- White rice (150g cooked)\n- Broccoli (100g)\n- *~550 kcal | 40g protein*\n\n### **Meal 4: Post-Workout (4:00 PM)**\n- Protein shake: Whey protein (30g), banana, oats (50g), milk (250ml)\n- *~400 kcal | 35g protein*\n\n### **Meal 5: Dinner (7:30 PM)**\n- Lean ground beef (180g)\n- Sweet potato (200g)\n- Mixed vegetables (150g)\n- Olive oil (1 tbsp)\n- *~650 kcal | 35g protein*\n\n---\n\n## **WEEKLY ROTATION TIPS**\n- Swap chicken for fish, turkey, or beef\n- Alternate rice with pasta, potatoes, or quinoa\n- Prep proteins in bulk on Sundays\n\n**Total: ~3,200 kcal | 155g protein**
    </plan>
    <plan_evaluation_comment>
        Plan is a solid, well-organized meal plan that successfully addresses the core requirements for muscle gain. The protein intake is excellent and properly distributed, meal timing supports training, and the format is clear and actionable. However, the carbohydrate target is slightly below the recommended range, which could impact energy availability and recovery. The fat intake, while adequate, could be better integrated throughout meals. The plan would benefit from slightly increased carbs (add ~25-50g) and more explicit healthy fat inclusion. Overall, this is a practical, implementable plan with minor optimization needed.
    </plan_evaluation_comment>
</example_plan>

"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages=messages, model=model)
    return output

In [58]:
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
You are an expert Nutrition doctors (clinical nutritionists/dietitians). Your task is to evaluate the following AI-generated 1 day meal plan for a single athlete.

Athlete details:
<bio_data>
{test_case["prompt_input"]}
</bio_data>

Meal plan to Evaluate:
<plan>
{output}
</plan>

Criteria you should use to evaluate the meal plan
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [59]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # Grade the output
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    score = model_score
    
    return {
        "test_case": test_case,
        "answer": output,
        "score": score,
        "reasoning": reasoning
    }

In [60]:
def run_evaluation(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
    
    return results

In [61]:
with open("ex04_dataset.json", "r") as f:
    dataset = json.load(f)

results = run_evaluation(dataset)

print(results)

Average score: 7.5
[{'test_case': {'prompt_input': {'height': '180 cm', 'weight': '75 kg', 'goal': 'Muscle gain', 'restrictions': 'None'}, 'solution_criteria': ['Meal plan should contain 4-5 meals distributed throughout the day', 'Daily caloric intake should be approximately 2,800-3,200 calories', 'Protein intake should be at least 150g (2g per kg body weight)', 'Include carbohydrates for energy (400-500g)', 'Include healthy fats (70-90g)', 'Meals should be practical and quick to prepare', 'Should include pre and post-workout nutrition', 'Format should be compact and easy to read']}, 'answer': '```\n# 7-Day Muscle Gain Meal Plan for 75kg Athlete\n\n**Daily Caloric Target:** ~3,200-3,400 kcal | **Protein:** ~150-160g | **Carbs:** ~400-420g | **Fat:** ~90-100g\n\n---\n\n## **DAILY MEAL STRUCTURE**\n\n### **Meal 1: Breakfast (7:00 AM)**\n- 3 whole eggs + 2 egg whites\n- 100g oatmeal with banana\n- 1.5 tbsp almond butter\n- *~700 kcal | 28g protein*\n\n### **Meal 2: Mid-Morning Snack (10:0